In [1]:
import os
import json
import pandas as pd
import subprocess
import pyfaidx
import pyfastx
import gzip

In [2]:
EVALUATOR_DATA_DIR = "./evaluator_data"

RAW_DATA_DIR = "./raw_data"
RAW_DATA_FILE_PATH = os.path.join(RAW_DATA_DIR, "GRCh38_ALL.tsv")

GENOME_FILE_PATH = os.path.join(EVALUATOR_DATA_DIR, "hg38.fa")

# Dictionary mapping cell types to their corresponding genetic elements
# Used for organizing and processing data specific to each cell line
ELEMENTS_BY_CELL_TYPE = {
    "HepG2": ["F9", "LDLR", "LDLR.2", "SORT1", "SORT1-flip", "SORT1.2"],
    "HEL92.1.7": ["GP1BA", "HBB", "HBG1", "BCL11A"],
    "HEK293T": ["HNF4A", "MSMB", "MYCrs6983267", "TERT-HEK"],
    "K562": ["PKLR-24h", "PKLR-48h"],
    "SF7996": ["TERT-GAa", "TERT-GBM", "TERT-GSc"],
    "HeLa": ["FOXE1"],
    "SK-MEL-28": ["IRF4"],
    "HaCaT": ["IRF6"],
    "LNCaP": ["MYCrs11986220"],
    "Neuro-2a": ["RET", "UC88"],
    "MIN6": ["TCF7L2", "ZFAND3"],
    "NIH-3T3": ["ZRSh-13", "ZRSh-13h2"],
}

In [3]:
# Check if evaluator data dir exists
if not os.path.exists(EVALUATOR_DATA_DIR):
    os.mkdir(EVALUATOR_DATA_DIR)

# Check if raw Kircher data exists:
if not os.path.exists(RAW_DATA_FILE_PATH):
    
    if not os.path.exists(RAW_DATA_DIR):
        os.mkdir(RAW_DATA_DIR)

    url = "https://raw.githubusercontent.com/visze/sequence_cnn_models/refs/heads/master/resources/satmut_examples/GRCh38_ALL.tsv"
    try:
        # Download raw Kircher data
        subprocess.run(["wget", url, "-O", RAW_DATA_FILE_PATH], check=True)
    except subprocess.CalledProcessError as e:
        raise IOError(f"Failed to download raw data file: {str(e)}") from e
else:
    print("Raw data file already exists. Skipping download.")

Raw data file already exists. Skipping download.


In [4]:
# Check if genome file exists, download if missing
if not os.path.exists(GENOME_FILE_PATH):
    url = "http://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz"
    try:
        # Download and decompress reference genome
        subprocess.run(["wget", url, "-O", f"{GENOME_FILE_PATH}.gz"], check=True)
        subprocess.run(["gunzip", GENOME_FILE_PATH + ".gz"], check=True)
    except subprocess.CalledProcessError as e:
        raise IOError(f"Failed to download/decompress FASTA file: {str(e)}") from e
else:
    print("FASTA file already exists. Skipping download.")

FASTA file already exists. Skipping download.


In [5]:
df = pd.read_csv(RAW_DATA_FILE_PATH, sep = "\t")

In [6]:
# Create mapping dict
cell_type_by_element = {}
for cell_type, elements in ELEMENTS_BY_CELL_TYPE.items():
    for element in elements:
        cell_type_by_element[element] = cell_type
 
# Create the cell type column
df["Cell_Type"] = df["Element"].map(cell_type_by_element)

df

,Chromosome,Position,Ref,Alt,Tags,DNA,RNA,Value,P-Value,Element,Cell_Type
0,2,60494940,C,-,32,577,1345,-0.34,0.00546,BCL11A,HEL92.1.7
1,2,60494940,C,A,146,2785,6772,-0.05,0.38889,BCL11A,HEL92.1.7
2,2,60494940,C,G,60,975,2436,-0.13,0.13721,BCL11A,HEL92.1.7
3,2,60494940,C,T,1084,8543,16057,-0.70,0.00000,BCL11A,HEL92.1.7
4,2,60494941,C,A,596,9425,23430,-0.08,0.00413,BCL11A,HEL92.1.7
...,...,...,...,...,...,...,...,...,...,...,...
44642,7,156791603,C,-,1447,11717,34300,0.04,0.11224,ZRSh-13h2,NIH-3T3
44643,7,156791603,C,A,25,107,309,-0.17,0.33374,ZRSh-13h2,NIH-3T3
44644,7,156791603,C,G,654,4748,14458,0.06,0.09092,ZRSh-13h2,NIH-3T3
44645,7,156791603,C,T,884,8072,24544,0.17,0.00000,ZRSh-13h2,NIH-3T3


In [7]:
# Add Name column (combination of Element_CellType_Position_Ref_to_Alt)
df["Name"] = df.apply(lambda x: f"{x.Element}_{x.Cell_Type}_{x.Position}_{x.Ref}_to_{x.Alt}", axis=1)

# Split by regulatory elements
elements = df['Element'].unique()

for element in elements:
    # Filter data for current element
    element_df = df[df['Element'] == element]
    cell_type = element_df["Cell_Type"].iloc[0]
    
    # Create subdirectory for cell type
    element_dir = os.path.join(EVALUATOR_DATA_DIR, cell_type)
    os.makedirs(element_dir, exist_ok=True)
    
    # Create sequences table
    sequences_df = element_df[["Name", "Chromosome", "Position", "Ref", "Alt", "Element", "Cell_Type", "Value"]].copy()
    sequences_file = os.path.join(element_dir, f"{element}_variants.tsv")
    sequences_df.to_csv(sequences_file, sep='\t', index=False)

In [8]:
# Read genome.fa
genome =  pyfaidx.Fasta(GENOME_FILE_PATH)

# Define region for each regulatory element
elements = df.groupby(['Element', 'Cell_Type', 'Chromosome'])

for (element, cell_type, chrom), group in elements:
    min_pos = group['Position'].min()
    max_pos = group['Position'].max()
    
    # Cut 1mb (±500kb)
    start = max(1, min_pos - 500000)
    end = max_pos + 500000

    chr = "chr" + chrom
    
    region_id = f"{element}_{cell_type}"
    output_dir = os.path.join(EVALUATOR_DATA_DIR, cell_type)
    os.makedirs(output_dir, exist_ok=True)
    
    # Extract the sequence
    try:
        sequence = genome[chr][start-1:end]  # -1 because pyfaidx 0-based
        with gzip.open(f"{output_dir}/{element}_sequence.fasta.gz", "wb") as f:
            f.write(f">{region_id} {chr}:{start}-{end}\n".encode('ascii'))
            f.write(str(sequence).encode('ascii') + b"\n")
        
        # Save metadata
        metadata = {
            "element": element,
            "cell_type": cell_type,
            "chromosome": str(chrom),
            "start": str(start),
            "end": str(end),
            "length": str(end - start + 1)
        }
        with open(f"{output_dir}/{element}_metadata.json", "w") as f:
            json.dump(metadata, f, indent=2)
            
    except Exception as e:
        print(f"Error extracting {region_id}: {e}")

In [9]:
# How to open .fasta.gz files
iref = pyfastx.Fastx("./evaluator_data/K562/PKLR-24h_sequence.fasta.gz", comment = True)

for name, seq, comment in iref:
    name = name + "_" + comment
    seq = seq

In [10]:
name

'PKLR-24h_K562_chr1:154801395-155801864'

In [11]:
seq[0:100]

'AGAGGGAAATAGATCAATGAGAAAGCCAGAGCCCAGAGAGCCTGTGCTGGCGGCTCTGAAGCAAGAACATCCATTCTTCCCCAAGAGCCTACGCCAAAAG'